### Imports

In [19]:
from abc import ABC, abstractmethod
from typing import List, Any, cast, Dict, Tuple, Set
from collections import defaultdict
import subprocess, re, json, pm4py, hashlib, os, random, time, csv, sys, math, shutil, subprocess, pathlib, warnings
from functools import lru_cache
from dataclasses import dataclass
from math import sqrt
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from graphviz import Digraph
from itertools import count
from IPython.display import display
warnings.filterwarnings('ignore')
from pathlib import Path

### Empty nodes counter

In [2]:
def enumerate_tau(node, counter):
    if (not hasattr(node, "children") or not node.children) and getattr(node, "label", None) is None:
        node.label = f"tau_{counter[0]}"
        counter[0] += 1

    if hasattr(node, "children") and node.children is not None:
        for child in node.children:
            enumerate_tau(child, counter)

none_counter = [1]

### Process tree

In [4]:
class ASTNode(ABC):
    @abstractmethod
    def to_string(self, depth: int = 0) -> str:
        pass

class LiteralASTNode(ASTNode):
    def __init__(self, label: str):
        self.label = label

    def to_string(self, depth: int = 0) -> str:
        indent = "    " * depth
        return f"{indent}Literal: '{self.label}'"

class OperatorASTNode(ASTNode):
    def __init__(self, operator: str, children: List[ASTNode]):
        self.operator = operator
        self.children = children

    def to_string(self, depth: int = 0) -> str:
        indent = "    " * depth
        child_str = "\n".join(child.to_string(depth + 1) for child in self.children)
        return f"{indent}Operator: {self.operator}\n{child_str}"

### Process tree to AST

Converts process tree to AST

In [5]:
def process_tree_to_ast(pt: Any) -> ASTNode:
    if hasattr(pt, "children") and pt.children:
        op = getattr(pt, "operator", None) or "Seq"
        children_ast = [process_tree_to_ast(c) for c in pt.children]
        return OperatorASTNode(operator=str(op), children=children_ast)
    else:
        label = getattr(pt, "label", None) or "tau"
        return LiteralASTNode(label=str(label))

### Visitor

In [6]:
class ASTVisitor(ABC):
    @abstractmethod
    def visit_literal(self, node: LiteralASTNode):
        pass

    @abstractmethod
    def visit_operator(self, node: OperatorASTNode):
        pass

    def visit(self, node: ASTNode) -> Any:
        if isinstance(node, LiteralASTNode):
            lit = cast(LiteralASTNode, node)
            return self.visit_literal(lit)
        elif isinstance(node, OperatorASTNode):
            op = cast(OperatorASTNode, node)
            return self.visit_operator(op)
        else:
            ...

### AST Visitor extension

Creates logical specification from AST

In [7]:
class LogicalSpecificationVisitor(ASTVisitor):
    def visit_literal(self, node: LiteralASTNode):
        return f"'{node.label}'"

    def visit_operator(self, node: OperatorASTNode):
        op = node.operator.lower()
        if op == "seq":
            if len(node.children) == 2:
                ini = self.visit(node.children[0])
                fin = self.visit(node.children[1])
                return [
                    f"{ini} => <> {fin}",
                    f"~{ini} => ~<> {fin}",
                    f"[]~({ini} & {fin})"
                ]
            return [
                f"{self.visit(node.children[i])} => <> {self.visit(node.children[i+1])}"
                for i in range(len(node.children) - 1)
            ]
        parts = []
        for c in node.children:
            r = self.visit(c)
            parts.append(r[0] if isinstance(r, list) else r)
        args_str = ", ".join(parts)
        return [f"{node.operator}({args_str})"]

visitor = LogicalSpecificationVisitor()

### Fromatting

Basic functions that help with activities

In [8]:
class Format:
    @staticmethod
    def lowercase_in_quotes(s: str) -> str:
        def _repl(m: re.Match) -> str:
            inner = m.group(1)
            return f"'{inner.lower()}'"
        return re.sub(r"'([^']*)'", _repl, s)

    @staticmethod
    def replace_parens_in_quotes(s: str) -> str:
        def _repl(m: re.Match) -> str:
            inner = m.group(1).replace('(', '-').replace(')', '-')
            return f"'{inner}'"
        return re.sub(r"'([^']*)'", _repl, s)

    @staticmethod
    def replace_colons_in_quotes(s: str) -> str:
        def _repl(m: re.Match) -> str:
            inner = m.group(1)
            return f"'{inner.replace(':', '_')}'"
        return re.sub(r"'([^']*)'", _repl, s)

    @staticmethod
    def replace_spaces_with_underscore(text: str) -> str:
        def replace_spaces(m: re.Match) -> str:
            return re.sub(r'\s+', '_', m.group(0))
        text_with_underscore = re.sub(r"'(.*?)'", replace_spaces, text)
        text_no_quotes = re.sub(r"'", '', text_with_underscore)
        return text_no_quotes

    @staticmethod
    def label_expressions(expression: str) -> str:
        labelled_expression = ""
        label_number = 0
        for c in expression:
            if c == '(':
                label_number += 1
                labelled_expression += f"({label_number}]"
            elif c == ')':
                labelled_expression += f"[{label_number})"
                label_number -= 1
            else:
                labelled_expression += c
        return labelled_expression

### Process tree adapter

Defines tools for argument extraction

In [9]:
class ProcessTreeAdapter:
    @staticmethod
    def extract_arguments_from_labelled_expression(labelled_expression):
        pattern_label_number = int(labelled_expression[labelled_expression.index(
            "(") + 1:labelled_expression.index("]")])
        trimmed_labelled_expression = labelled_expression[labelled_expression.index("]") + 1: list(re.finditer(r'\[', labelled_expression))[-1].start()]
        split = trimmed_labelled_expression.split(",")
        arguments = []
        brackets_counter = 0
        temp_arg = ""
        for s in split:
            brackets_counter += s.count('(')
            brackets_counter -= s.count(')')
            temp_arg += s + ","
            if brackets_counter == 0:
                temp_arg = temp_arg[:-1]
                arguments.append(temp_arg)
                temp_arg = ""
        return arguments, pattern_label_number

    @staticmethod
    def find_symbol(labelled_expression):
        pattern = r'^[^()]*'
        match = re.match(pattern, labelled_expression)
        if match:
            return re.sub(r'\s+', '', match.group())
        else:
            raise Exception("No match")

    @staticmethod
    def replace_symbol_with_name(labelled_pattern_expression, pattern_label_number, old_symbol, new_name):
        pattern = rf"{re.escape(old_symbol)}\(\s*{pattern_label_number}\s*\]"
        final_name = f"{new_name}({pattern_label_number}]"
        replaced_string = re.sub(pattern, final_name, labelled_pattern_expression, count=1)

        return replaced_string

    @staticmethod
    def get_highest_label(labelledExpression: str) -> int:
        maxLabel = -1
        active = False
        sb = ""
        for c in labelledExpression:
            if c == '(':
                active = True
            elif c == ']':
                if int(sb) > maxLabel:
                    maxLabel = int(sb)
                sb = ""
                active = False
            elif active:
                sb += c
        return maxLabel

### Pattern operators

In [10]:
class Sequence:
    _counter = defaultdict(int)

    @staticmethod
    def change_symbol_into_name(labelled_expression, pattern_label_number):
        n = len(labelled_expression)
        if not (2 <= n <= 15):
            raise Exception(f"Pattern for Seq{n} does not exist")

        Sequence._counter[n] += 1

        pattern_name = f'Seq{n}'
        old = f"({pattern_label_number}]" + ",".join(labelled_expression) + f"[{pattern_label_number})"
        new = old

        return pattern_name, old, new


class Loop:
    _counter = defaultdict(int)

    @staticmethod
    def change_symbol_into_name(labelled_expression, pattern_label_number):
        n = len(labelled_expression)
        if not (2 <= n <= 30):
            raise Exception(f"Pattern for Loop{n} does not exist")

        Loop._counter[n] += 1
        idx = Loop._counter[n]

        pattern_name = f'Loop{n}'
        old = "(" + str(pattern_label_number) + "]" + ",".join(labelled_expression) + "[" + str(pattern_label_number) + ")"

        start = f'l{n}_s_{idx}'
        end   = f'l{n}_e_{idx}'
        labelled_expression = [start] + labelled_expression + [end]

        new = "(" + str(pattern_label_number) + "]" + ",".join(labelled_expression) + "[" + str(pattern_label_number) + ")"
        return pattern_name, old, new


class ExclusiveChoice:
    _counter = defaultdict(int)

    @staticmethod
    def change_symbol_into_name(labelled_expression, pattern_label_number):
        n = len(labelled_expression)
        if not (2 <= n <= 30):
            raise Exception(f"Pattern for Xor{n} does not exist")

        ExclusiveChoice._counter[n] += 1
        idx = ExclusiveChoice._counter[n]
        pattern_name = f'Xor{n}'
        labelled_expression_to_replace = "(" + str(pattern_label_number) + "]" + ",".join(
            labelled_expression) + "[" + str(pattern_label_number) + ")"

        start = f'x{n}_s_{idx}'
        end   = f'x{n}_e_{idx}'
        labelled_expression = [start] + labelled_expression + [end]

        new_labelled_expression = "(" + str(pattern_label_number) + "]" + ",".join(
            labelled_expression) + "[" + str(pattern_label_number) + ")"
        return pattern_name, labelled_expression_to_replace, new_labelled_expression


class Parallelism:
    _counter = defaultdict(int)

    @staticmethod
    def change_symbol_into_name(labelled_expression, pattern_label_number):
        n = len(labelled_expression)

        if not (2 <= n <= 30):
            raise Exception(f"Pattern for And{n} does not exist")

        Parallelism._counter[n] += 1
        idx = Parallelism._counter[n]

        pattern_name = f'And{n}'
        labelled_expression_to_replace = "(" + str(pattern_label_number) + "]" + ",".join(
            labelled_expression) + "[" + str(pattern_label_number) + ")"

        start = f'a{n}_s_{idx}'
        end   = f'a{n}_e_{idx}'
        labelled_expression = [start] + labelled_expression + [end]

        new_labelled_expression = "(" + str(pattern_label_number) + "]" + ",".join(
            labelled_expression) + "[" + str(pattern_label_number) + ")"
        return pattern_name, labelled_expression_to_replace, new_labelled_expression

### Pattern Generator

Identifies operators

In [12]:
class PatternExpressionGenerator:
    def __init__(self, converted_expression):
        self.converted_expression = converted_expression

    def add_approved_workflow_patterns(self, expression):
        if expression is None or isinstance(expression, list):
            return

        symbol = ProcessTreeAdapter.find_symbol(expression)
        up = symbol.upper()
        if up.startswith("SEQ"):
            symbol = "->"
        elif up.startswith("AND"):
            symbol = "+"
        elif up.startswith("XOR"):
            symbol = "X"
        elif up.startswith("LOOP"):
            symbol = "*"

        pattern = r'>|X|\+|\*'
        matches = re.findall(pattern, expression)

        if len(matches) != 0:
            arguments = ProcessTreeAdapter.extract_arguments_from_labelled_expression(
                expression)
            expression = arguments[0]
            pattern_label_number = arguments[1]

            if symbol == '->' or symbol.upper().startswith("SEQ"):
                new_name = Sequence.change_symbol_into_name(expression, pattern_label_number)
                self.converted_expression = ProcessTreeAdapter.replace_symbol_with_name(
                    self.converted_expression, pattern_label_number, symbol, new_name[0])
                expr_tmp = self.converted_expression
                self.converted_expression = expr_tmp.replace(new_name[1], new_name[2])

            elif symbol == '*' or symbol.upper().startswith("LOOP"):
                new_name = Loop.change_symbol_into_name(
                    expression, pattern_label_number)
                self.converted_expression = ProcessTreeAdapter.replace_symbol_with_name(
                    self.converted_expression, pattern_label_number, symbol, new_name[0])
                self.converted_expression = self.converted_expression.replace(
                    new_name[1], new_name[2])

            elif symbol == '+' or symbol.upper().startswith("AND"):
                new_name = Parallelism.change_symbol_into_name(
                    expression, pattern_label_number)
                self.converted_expression = ProcessTreeAdapter.replace_symbol_with_name(
                    self.converted_expression, pattern_label_number, symbol, new_name[0])
                self.converted_expression = self.converted_expression.replace(
                    new_name[1], new_name[2])

            elif symbol == 'X' or symbol.upper().startswith("XOR"):
                new_name = ExclusiveChoice.change_symbol_into_name(
                    expression, pattern_label_number)
                self.converted_expression = ProcessTreeAdapter.replace_symbol_with_name(
                    self.converted_expression, pattern_label_number, symbol, new_name[0])
                self.converted_expression = self.converted_expression.replace(
                    new_name[1], new_name[2])

            else:
                raise Exception(f"Pattern for {symbol} does not exist")

        return expression

    def get_converted_expression(self):
        return self.converted_expression

### Get expression


Orchestrates the process of recursively converting a labeled expression to a named expression

In [13]:
class GetPatternExpression:

    @staticmethod
    def process_patterns(pattern_list: list, instance) -> list:
        new_pattern_list = []
        for pattern in pattern_list:
            new_pattern = instance.add_approved_workflow_patterns(pattern)
            if isinstance(new_pattern, list):
                new_pattern = GetPatternExpression.process_patterns(new_pattern, instance)
            new_pattern_list.append(new_pattern)
        return new_pattern_list

    @staticmethod
    def recursive_process(pattern_list: list, instance, depth: int) -> list:
        if depth <= 0:
            return pattern_list
        pattern_list = GetPatternExpression.process_patterns(pattern_list, instance)
        return GetPatternExpression.recursive_process(pattern_list, instance, depth - 1)

    @staticmethod
    def get_pattern_expression(labelled_pattern_expression: str) -> str:
        pattern_list = [labelled_pattern_expression]
        generator = PatternExpressionGenerator(labelled_pattern_expression)
        max_label = ProcessTreeAdapter.get_highest_label(labelled_pattern_expression)
        GetPatternExpression.recursive_process(pattern_list, generator, max_label)
        return generator.get_converted_expression()

### Patterns for workflows

In [14]:
class WorkflowPatternTemplate:
    def __init__(self, name, number_of_arguments, rules):
        self.name = name
        self.number_of_arguments = number_of_arguments
        self.rules = rules

    @staticmethod
    def load_pattern_property_set(path_to_pattern_rules_file):
        with open(path_to_pattern_rules_file, 'r') as file:
            e_data = json.load(file)
            pattern_property_set = []
            for workflow_pattern_template_name, pattern_descr_json_object in e_data.items():
                number_of_arguments = pattern_descr_json_object["number of args"]
                rules = pattern_descr_json_object["rules"]
                workflow_pattern_template = WorkflowPatternTemplate(
                    workflow_pattern_template_name, number_of_arguments, rules)
                pattern_property_set.append(workflow_pattern_template)
            return pattern_property_set

    def get_name(self):
        return self.name

    def set_name(self, name):
        self.name = name

    def get_number_of_arguments(self):
        return self.number_of_arguments

    def set_number_of_arguments(self, number_of_arguments):
        self.number_of_arguments = number_of_arguments

    def get_rules(self):
        return self.rules

    def set_rules(self, rules):
        self.rules = rules

### Workflow patterns

Represents a specific instance of a workflow pattern

In [15]:
class WorkflowPattern:
    def __init__(self, workflow_pattern_template, pattern_arguments):
        self.workflow_pattern_template = workflow_pattern_template
        self.pattern_arguments = pattern_arguments

    @staticmethod
    def get_workflow_pattern_from_expression(pattern_expression, pattern_property_set):
        workflow_name = pattern_expression[:pattern_expression.index("(")]
        workflow_pattern_template = next(
            (template for template in pattern_property_set if template.get_name() == workflow_name), None)

        if workflow_pattern_template is None:
            raise Exception("Workflow pattern template not found! Workflow name: " + workflow_name)
        pattern_arguments = WorkflowPattern.extract_arguments_from_labelled_expression(
            pattern_expression, pattern_property_set)
        return WorkflowPattern(workflow_pattern_template, pattern_arguments)

    @staticmethod
    def extract_arguments_from_labelled_expression(labelled_expression, pattern_property_set):
        workflow_name = labelled_expression[:labelled_expression.index("(")]
        workflow_pattern_template = next(
            (template for template in pattern_property_set if template.get_name() == workflow_name), None)
        if workflow_pattern_template is None:
            raise Exception("Workflow pattern template not found!")

        number_of_arguments = int(workflow_pattern_template.get_number_of_arguments())
        pattern_label_number = int(labelled_expression[labelled_expression.index(
            "(") + 1:labelled_expression.index("]")])
        trimmed_labelled_expression = labelled_expression[labelled_expression.index("]") + 1 : list(re.finditer(r'\[', labelled_expression))[-1].start()]

        split = trimmed_labelled_expression.split(",")
        arguments = []
        brackets_counter = 0
        temp_arg = ""
        for s in split:
            brackets_counter += s.count('(')
            brackets_counter -= s.count(')')
            temp_arg += s + ","
            if brackets_counter == 0:
                temp_arg = temp_arg[:-1]
                arguments.append(temp_arg)
                temp_arg = ""

        if len(arguments) != number_of_arguments:
            print(labelled_expression)
            print(pattern_label_number)
            raise Exception("Too much arguments")
        return arguments

    @staticmethod
    def count_occurrence_of_char(string, char):
        return string.count(char)

    @staticmethod
    def is_not_atomic(argument):
        return "=>" in argument or "|" in argument or "^" in argument or "]" in argument

    def get_workflow_pattern_template(self):
        return self.workflow_pattern_template

    def set_workflow_pattern_template(self, workflow_pattern_template):
        self.workflow_pattern_template = workflow_pattern_template

    def get_workflow_pattern_filled_rules(self):
        if len(self.pattern_arguments) > 0:
            outcomes = []
            for outcome in self.workflow_pattern_template.get_rules():
                outcome_with_params = outcome
                for i, arg in enumerate(self.pattern_arguments):
                    outcome_with_params = outcome_with_params.replace(
                        "arg" + str(i), arg)
                outcomes.append(outcome_with_params)
            return outcomes
        else:
            raise Exception(
                "No arguments for the given pattern in the expression")

    def get_pattern_arguments(self):
        return self.pattern_arguments

    def set_pattern_arguments(self, pattern_arguments):
        self.pattern_arguments = pattern_arguments

### Consolidated Expression generator

In [16]:
class CalculatingConsolidatedExpression:

    @staticmethod
    def generate_consolidated_expression(pattern_expression: str, expression_type: str, pattern_property_set: List[WorkflowPatternTemplate]) -> str:

        if expression_type not in ("ini", "fin"):
            raise Exception("Type must equal 'ini' or 'fin'!")

        workflow_pattern = WorkflowPattern.get_workflow_pattern_from_expression(
            pattern_expression, pattern_property_set)
        rules_with_atomic_activities = workflow_pattern.get_workflow_pattern_filled_rules()
        ini = rules_with_atomic_activities[0]
        fin = rules_with_atomic_activities[1]

        if expression_type == "ini":
            ex = ini
        else:
            ex = fin

        expression_arguments = WorkflowPattern.extract_arguments_from_labelled_expression(
            pattern_expression, pattern_property_set)
        for argument in expression_arguments:
            if WorkflowPattern.is_not_atomic(argument):
                inner_consolidated_expression = CalculatingConsolidatedExpression.generate_consolidated_expression(
                    argument, expression_type, pattern_property_set)

                ex = ex.replace(argument, inner_consolidated_expression)
        return ex

### Generating Logical Specification

In [17]:
class GeneratingLogicalSpecifications:

    @staticmethod
    def generate_logical_specifications(pattern_expression: str, pattern_property_set: List[WorkflowPatternTemplate], verbose=False) -> str:
        logical_specification = []
        labelled_expression = pattern_expression
        highest_label_number = ProcessTreeAdapter.get_highest_label(
            labelled_expression)
        for l in range(highest_label_number, 0, -1):
            c = 1
            pat = GeneratingLogicalSpecifications.get_pat(
                labelled_expression, l, c, pattern_property_set)
            while pat is not None:
                L2 = pat.get_workflow_pattern_filled_rules()
                L2 = L2[2:]
                for arg in pat.get_pattern_arguments():
                    if WorkflowPattern.is_not_atomic(arg):
                        cons = CalculatingConsolidatedExpression.generate_consolidated_expression(
                            arg, "ini", pattern_property_set) + " | " + CalculatingConsolidatedExpression.generate_consolidated_expression(arg, "fin", pattern_property_set)
                        L2_cons = [outcome.replace(arg, cons)
                                   for outcome in L2]
                        L2 = L2_cons
                c += 1
                logical_specification.extend(L2)
                pat = GeneratingLogicalSpecifications.get_pat(
                    labelled_expression, l, c, pattern_property_set)

        logical_specification = list(set(logical_specification))
        connected_string = ""
        if verbose:
            print("\nResult: ")
        for l_value in logical_specification:
            connected_string += l_value + "\n"
            if verbose:
                print(l_value)
        return connected_string

    @staticmethod
    def get_pat(labelled_expression: str, l: int, c: int, pattern_property_set: List[WorkflowPatternTemplate]) -> Any:
        entry_occurrences = labelled_expression.count("(" + str(l) + "]")
        end_occurrences = labelled_expression.count("[" + str(l) + ")")
        if entry_occurrences != end_occurrences:
            raise Exception("(" + str(l) + "] not equal [" + str(l) + ")")

        if entry_occurrences < c:
            return None

        expression_split_by_entry = re.split(rf"\({l}\]", labelled_expression)
        pattern_content = re.split(
            rf"\[{l}\)", expression_split_by_entry[c])[0]
        split_by_bracket = re.split(r"\]", expression_split_by_entry[c - 1])
        workflow_name = re.split(r",", split_by_bracket[-1])[-1]
        workflow_exp = workflow_name + f"({l}]" + pattern_content + f"[{l})"
        return WorkflowPattern.get_workflow_pattern_from_expression(workflow_exp, pattern_property_set)

### Ruleset path for expression

In [18]:
PATTERN_RULES_PATH = "../Data/patterns.json"
ltl_pattern_property_set = WorkflowPatternTemplate.load_pattern_property_set(
    PATTERN_RULES_PATH)

def get_results(pattern_expression):
    ini = CalculatingConsolidatedExpression.generate_consolidated_expression(
        pattern_expression.replace(" ", ""), "ini", ltl_pattern_property_set)
    print("ini: " + ini)
    fin = CalculatingConsolidatedExpression.generate_consolidated_expression(
        pattern_expression.replace(" ", ""), "fin", ltl_pattern_property_set)
    print("fin: " + fin)

    return GeneratingLogicalSpecifications.generate_logical_specifications(
        pattern_expression.replace(" ", ""), ltl_pattern_property_set)

### Vampire integration

In [43]:
VAMPIRE_TIME_LIMIT_S = 2
VAMPIRE_EXTRA_ARGS = ["--mode", "casc"]

def _find_vampire() -> str:
    p = shutil.which("vampire")
    if p:
        return p

    candidates = [
        "~/.local/bin/vampire",
        "~/vampire/bin/vampire",
        "~/vampire/bin/vampire_rel",
        "~/vampire",
    ]
    for c in candidates:
        cp = Path(os.path.expanduser(c))
        if cp.exists() and os.access(cp, os.X_OK):
            return str(cp)

    raise FileNotFoundError(
        "No 'vampire'. Try better."
    )

def run_vampire(tptp_file_path: str, verbose: bool = False):
    tptp = Path(tptp_file_path).expanduser()
    if not tptp.exists():
        if verbose:
            print(f"Plik TPTP nie istnieje: {tptp.resolve()}")
        return False, ""

    try:
        vampire_bin = _find_vampire()
        env = os.environ.copy()
        vdir = str(Path(vampire_bin).parent)
        env["PATH"] = vdir + os.pathsep + env.get("PATH", "")

        result = subprocess.run(
            [vampire_bin, "--input_syntax", "tptp", str(tptp)],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            env=env,
        )

        if verbose:
            print("Vampire Result:")
            print(result.stdout)
            if result.stderr:
                print("Errors:")
                print(result.stderr)

        return True, result.stdout

    except subprocess.CalledProcessError as e:
        if verbose:
            print("Error during vampire run:")
            print(f"Exit code: {e.returncode}")
            print(f"Out:\n{e.stdout}")
            print(f"Errors:\n{e.stderr}")
        return False, e.stdout if e.stdout else ""

    except FileNotFoundError as e:
        if verbose:
            print(f"No file: {e}")
        return False, ""


### TPTP transformer

Transforms formulas into TPTP format

In [21]:
_exist_pattern     = re.compile(r"(?i)Exist\s*\((.*?)\)", re.DOTALL)
_op_pattern        = re.compile(r"\s*(&|\||=>)\s*")
_atom_pattern      = re.compile(r"\b[A-Za-z][A-Za-z0-9_]*\b")
_pred_call_pattern = re.compile(r"([a-z][0-9a-z_]*\(\s*X\s*\))")

def transform_to_tptp(input_str: str) -> str:
    def inline_existentials(formula: str) -> str:
        prev = None
        while prev != formula:
            prev = formula
            formula = _exist_pattern.sub(
                lambda m: f"?[X]: ( {inline_existentials(m.group(1).strip().rstrip('.-'))} )",
                formula
            )
        return formula

    lines = [ln.strip() for ln in input_str.splitlines() if ln.strip()]
    result = []

    for idx, line in enumerate(lines, start=1):
        name    = f"f{idx}"
        content = line.strip()

        quant = ""
        if content.startswith("ForAll"):
            quant   = "!"
            content = content[len("ForAll"):].strip()
        elif content.startswith("Exist"):
            quant   = "?"
            content = content[len("Exist"):].strip()

        if content.startswith("(") and content.endswith(")"):
            content = content[1:-1].strip()

        content = inline_existentials(content)

        content = content.replace("^", "&")
        content = _op_pattern.sub(lambda m: f" {m.group(1)} ", content)
        content = " ".join(content.split())

        content = content.replace(".", "")
        content = content.replace("-", "")

        def atom_to_pred(m: re.Match) -> str:
            tok = m.group(0)
            if tok in {"&","|","=>","?","!","[","]",":","X"}:
                return tok
            return f"{tok.lower()}(X)"

        expr = _atom_pattern.sub(atom_to_pred, content)
        expr = _pred_call_pattern.sub(r"(\1)", expr)

        prefix = f"{quant}[X]:" if quant in ("!","?") else ""

        result.append(f"fof({name}, axiom, {prefix} {expr}).")

    return "\n".join(result)

### Data config

In [23]:
LOG_PATHS: Dict[str, str] = {
    "running_example": "../Data/running-example.xes",
    "hospital_billing": "../Data/Hospital Billing - Event Log.xes",
    "bpi_2012": "../Data/BPI_Challenge_2012.xes",
}

NOISE_LEVELS = [0.0, 0.25, 0.5, 1.0]

### PM4Py helpers

In [22]:
def load_event_log(path: str):
    return pm4py.read_xes(path)

def discover_tree_inductive(event_log, noise: float):
    return pm4py.discover_process_tree_inductive(event_log, noise_threshold=noise)

def assign_tau_labels(tree: Any):
    counter = [1]
    enumerate_tau(tree, counter)
    return tree

def count_nodes(tree: Any) -> int:
    if not hasattr(tree, "children") or not tree.children:
        return 1
    return 1 + sum(count_nodes(c) for c in tree.children)

### Loading event log

In [24]:
EVENT_LOGS: Dict[str, Any] = {}
for name, path in LOG_PATHS.items():
    EVENT_LOGS[name] = load_event_log(path)

{ k: len(v) for k, v in EVENT_LOGS.items() }

parsing log, completed traces ::   0%|          | 0/6 [00:00<?, ?it/s]

parsing log, completed traces ::   0%|          | 0/100000 [00:00<?, ?it/s]

parsing log, completed traces ::   0%|          | 0/13087 [00:00<?, ?it/s]

{'running_example': 42, 'hospital_billing': 451359, 'bpi_2012': 262200}

### Discover process trees

In [25]:
TREES: Dict[Tuple[str, float], Any] = {}

for log_name, log_obj in EVENT_LOGS.items():
    for noise in NOISE_LEVELS:
        tree = discover_tree_inductive(log_obj, noise=noise)
        tree = assign_tau_labels(tree)
        TREES[(log_name, noise)] = tree

### Summary

In [26]:
SUMMARY: Dict[str, Dict[float, int]] = {}
for (log_name, noise), tree in TREES.items():
    SUMMARY.setdefault(log_name, {})[noise] = count_nodes(tree)

SUMMARY

{'running_example': {0.0: 14, 0.25: 14, 0.5: 14, 1.0: 14},
 'hospital_billing': {0.0: 93, 0.25: 71, 0.5: 61, 1.0: 41},
 'bpi_2012': {0.0: 99, 0.25: 78, 0.5: 69, 1.0: 24}}

### Conversion

- Map pm4py operator to symbol
- Build raw pattern expression from process tree

In [27]:
def _op_symbol(op_obj) -> str:
    s = str(op_obj).lower()
    if "loop" in s: return "*"
    if "parallel" in s or "and" in s: return "+"
    if "xor" in s or "exclusive" in s: return "X"
    # default: sequence
    return "->"

def tree_to_pattern_expr(node) -> str:
    has_children = hasattr(node, "children") and node.children
    if not has_children:
        label = getattr(node, "label", "tau")
        return f"'{label}'"
    sym = _op_symbol(getattr(node, "operator", "Seq"))
    args = [tree_to_pattern_expr(c) for c in node.children]
    inner = ",".join(args)
    return f"{sym}({inner})"

### Conversion pipeline

In [28]:
def sanitize_pattern_expression(raw_expr: str) -> str:
    s1 = Format.lowercase_in_quotes(raw_expr)
    s2 = Format.replace_parens_in_quotes(s1)
    s3 = Format.replace_colons_in_quotes(s2)
    s4 = Format.replace_spaces_with_underscore(s3)
    return s4

def label_and_convert(expr_noquotes: str) -> str:
    labelled = Format.label_expressions(expr_noquotes)
    return GetPatternExpression.get_pattern_expression(labelled)

def tree_to_named_pattern_expression(tree) -> str:
    raw = tree_to_pattern_expr(tree)
    clean = sanitize_pattern_expression(raw)
    named = label_and_convert(clean)
    return named

### Pattern templates loader

In [29]:
def load_pattern_templates(path: str = PATTERN_RULES_PATH):
    return WorkflowPatternTemplate.load_pattern_property_set(path)

def build_full_spec(named_pattern_expr: str, pattern_templates) -> str:
    return GeneratingLogicalSpecifications.generate_logical_specifications(
        named_pattern_expr, pattern_templates, verbose=False
    )

### Summarize

In [31]:
PATTERNS = {}
SPECS = {}
TEMPLATES = load_pattern_templates()

for key, tree in TREES.items():
    named_expr = tree_to_named_pattern_expression(tree)
    PATTERNS[key] = named_expr
    SPECS[key] = build_full_spec(named_expr, TEMPLATES)

### Prover helper

In [32]:
def _count_lines(s: str) -> int:
    return sum(1 for ln in s.splitlines() if ln.strip())

SPEC_LINES_SUMMARY = {}
for (log_name, noise), spec in SPECS.items():
    SPEC_LINES_SUMMARY.setdefault(log_name, {})[noise] = _count_lines(spec)

SPEC_LINES_SUMMARY

{'running_example': {0.0: 21, 0.25: 21, 0.5: 21, 1.0: 21},
 'hospital_billing': {0.0: 144, 0.25: 115, 0.5: 106, 1.0: 144},
 'bpi_2012': {0.0: 161, 0.25: 132, 0.5: 117, 1.0: 60}}

### Helper

Compute ini/fin for a named expression; convert []/<> wrappers to FO-friendly forms

In [33]:
def extract_ini_fin(named_pattern_expr: str, pattern_templates) -> tuple[str, str]:
    ini = CalculatingConsolidatedExpression.generate_consolidated_expression(
        named_pattern_expr, "ini", pattern_templates
    )
    fin = CalculatingConsolidatedExpression.generate_consolidated_expression(
        named_pattern_expr, "fin", pattern_templates
    )
    return ini, fin

def temporal_to_fo_wrappers(formula: str) -> str:
    s = formula
    s = re.sub(r"\s+", " ", s).strip()

    def _wrap_all(match: re.Match) -> str:
        inner = match.group(1)
        return f"ForAll({inner})"

    def _wrap_exist(match: re.Match) -> str:
        inner = match.group(1)
        return f"Exist({inner})"

    s = re.sub(r"\[\]\s*\((.*?)\)", _wrap_all, s)
    s = re.sub(r"<>\s*\((.*?)\)", _wrap_exist, s)
    return s

### Prover glue

Parse SZS status, cache results, build TPTP, and evaluate SAT/entailment

In [35]:
def parse_vampire_status(raw_output: str) -> str:
    m = re.search(r"SZS status\s+([A-Za-z]+)", raw_output)
    return m.group(1) if m else "Unknown"

OUT_DIR = "../Docs/Problems/out"
os.makedirs(OUT_DIR, exist_ok=True)

_PROVER_CACHE: dict[tuple[str, str], int] = {}

def _hash_key(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def tptp_from_spec(spec_text: str) -> str:
    return transform_to_tptp(spec_text)

def tptp_from_spec_and_conjecture(spec_text: str, conjecture_text: str) -> str:
    t_axioms = transform_to_tptp(spec_text)
    t_conj = transform_to_tptp(conjecture_text).strip().splitlines()
    t_conj = [ln.replace(", axiom,", ", conjecture,") for ln in t_conj]
    return t_axioms.rstrip() + "\n" + "\n".join(t_conj) + "\n"

def eval_satisfiable(spec_text: str, *, verbose: bool = False) -> int:
    key = (_hash_key(spec_text), "sat")
    if key in _PROVER_CACHE:
        return _PROVER_CACHE[key]
    tptp_content = tptp_from_spec(spec_text)
    path = os.path.join(OUT_DIR, "out_sat_tmp.p")
    with open(path, "w", encoding="utf-8") as f:
        f.write(tptp_content)
    ok, out = run_vampire(path, verbose=verbose)
    status = parse_vampire_status(out if ok else "")
    val = 1 if status.lower() == "satisfiable" else 0
    _PROVER_CACHE[key] = val
    return val

def eval_entails(spec_text: str, property_text: str, *, verbose: bool = False) -> int:
    key = (_hash_key(spec_text + "\n#\n" + property_text), "entails")
    if key in _PROVER_CACHE:
        return _PROVER_CACHE[key]
    tptp_content = tptp_from_spec_and_conjecture(spec_text, property_text)
    path = os.path.join(OUT_DIR, "out_entails_tmp.p")
    with open(path, "w", encoding="utf-8") as f:
        f.write(tptp_content)
    ok, out = run_vampire(path, verbose=verbose)
    status = parse_vampire_status(out if ok else "")
    val = 1 if status.lower() == "theorem" else 0
    _PROVER_CACHE[key] = val
    return val

### Properties builder 

Build liveness/safety variants and evaluate them (or satisfiability) via the prover

In [36]:
def build_liveness_property(ini: str, fin: str, pattern: str = "standard") -> str:
    pattern = pattern.lower()
    if pattern == "standard" or pattern == "response":
        return temporal_to_fo_wrappers(f"[] ({ini} => <> ({fin}))")
    elif pattern == "eventual":
        return temporal_to_fo_wrappers(f"<> ({fin})")
    else:
        raise ValueError(f"Unknown liveness pattern: {pattern}")

def build_safety_property(ini: str, fin: str, pattern: str = "standard") -> str:
    pattern = pattern.lower()
    if pattern == "standard" or pattern == "mutual_exclusion":
        return temporal_to_fo_wrappers(f"[] ~({ini} & {fin})")
    elif pattern == "precedence":
        return temporal_to_fo_wrappers(f"[] ({fin} => ({ini} WU {fin}))")
    elif pattern == "absence":
        return temporal_to_fo_wrappers(f"[] ~({ini})")
    elif pattern == "invariant":
        return temporal_to_fo_wrappers(f"[] ({ini} => {fin})")
    elif pattern == "until":
        return temporal_to_fo_wrappers(f"[] ({ini} => ({ini} | <> {fin}))")
    else:
        raise ValueError(f"Unknown safety pattern: {pattern}")

def evaluate_property(spec_text: str, property_type: str, ini: str | None = None, fin: str | None = None, 
                     pattern: str = "standard", *, verbose: bool = False) -> int:
    p = property_type.lower()
    
    if p == "satisfiability":
        return eval_satisfiable(spec_text, verbose=verbose)
    elif p == "liveness":
        assert ini is not None and fin is not None, "ini/fin required for liveness"
        prop = build_liveness_property(ini, fin, pattern)
        return eval_entails(spec_text, prop, verbose=verbose)
    elif p == "safety":
        assert ini is not None and fin is not None, "ini/fin required for safety"
        prop = build_safety_property(ini, fin, pattern)
        return eval_entails(spec_text, prop, verbose=verbose)
    else:
        raise ValueError(f"Unknown property type: {property_type}. Supported: satisfiability, liveness, safety")


### Precompute ini/fin formulas

In [37]:
INI_FIN: dict[tuple[str, float], tuple[str, str]] = {}

for key, named_expr in PATTERNS.items():
    ini, fin = extract_ini_fin(named_expr, TEMPLATES)
    INI_FIN[key] = (ini, fin)

### Smoke test

In [44]:
_sample_key = next(iter(SPECS.keys()))
_sample_spec = SPECS[_sample_key]
_sample_ini, _sample_fin = INI_FIN[_sample_key]

_sample_results = {
    "satisfiability": evaluate_property(_sample_spec, "satisfiability"),
    "liveness": evaluate_property(_sample_spec, "liveness", _sample_ini, _sample_fin),
    "safety": evaluate_property(_sample_spec, "safety", _sample_ini, _sample_fin),
}
_sample_key, _sample_results

(('running_example', 0.0), {'satisfiability': 1, 'liveness': 1, 'safety': 0})

### Players

Pattern nodes mapped to ids Name@Label; helpers to extract by label and list all players

In [45]:
@dataclass(frozen=True)
class Player:
    id: str
    name: str
    label: int
    snippet: str

def _extract_pattern_by_label(named_expr: str, l: int) -> tuple[str, str] | None:
    entry_occ = named_expr.count(f"({l}]")
    end_occ = named_expr.count(f"[{l})")
    if entry_occ != end_occ or entry_occ == 0:
        return None
    parts = re.split(rf"\({l}\]", named_expr)
    pattern_content = re.split(rf"\[{l}\)", parts[1])[0]
    prefix = re.split(r"\]", parts[0])[-1]
    workflow_name = re.split(r",", prefix)[-1]
    snippet = f"{workflow_name}({l}]{pattern_content}[{l})"
    return workflow_name, snippet

def list_players_from_expression(named_expr: str) -> list[Player]:
    players: list[Player] = []
    highest = ProcessTreeAdapter.get_highest_label(named_expr)
    for l in range(highest, 0, -1):
        found = _extract_pattern_by_label(named_expr, l)
        if not found:
            continue
        name, snippet = found
        pid = f"{name}@{l}"
        players.append(Player(id=pid, name=name, label=l, snippet=snippet))
    players.sort(key=lambda p: p.label)
    return players